# RAG on Azure — Day 7: Consolidation

**Use case:** turn six days of standalone notebooks into a real project. Refactor duplicated plumbing into a shared `common/` module, write READMEs, add config files, and push to GitHub.

- A `common/` Python module with `config`, `clients`, `ingestion`, `retrieval`, and `generation`.
- Every prior day's notebook can collapse ~80 lines of setup into 3 lines of imports.
- Top-level `README.md`, `requirements.txt`, `.env.example`, and `.gitignore`.
- A repository that reads as a portfolio piece, not a stack of scripts.

## 0. Install dependencies

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import openai, httpx
print("openai :", openai.__version__)
print("httpx  :", httpx.__version__)

openai : 2.41.0
httpx  : 0.28.1


In [1]:
%pip install -q --upgrade "openai>=1.55.3"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. The new setup — 3 lines of imports, 2 lines of config

In [3]:
from pathlib import Path
import sys
sys.path.insert(0, "..")                # so the notebook can find common/ in the repo root

from common import load_config, get_openai_client, get_search_client
from common import retrieve, answer, answer_with_refusal, verify_all
config = load_config(Path(r"C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env"))
openai = get_openai_client(config)
print("config OK. Chat:", config["chat_deployment"], "| Embedding:", config["embedding_deployment"])

config OK. Chat: gpt-4o | Embedding: text-embedding-3-large


## 2. End-to-end demo using the shared module

In [4]:
INDEX_NAME = "rag-handbook-day6"   # the index built on Day 6
search = get_search_client(config, INDEX_NAME)

print(f"Connected to '{INDEX_NAME}' — {search.get_document_count()} documents.\n")

Connected to 'rag-handbook-day6' — 24 documents.



In [5]:
question = "How many PTO days do full-time employees get?"

hits = retrieve(
    search, openai,
    query=question,
    embedding_deployment=config["embedding_deployment"],
    k=5,
    select=["content", "source", "page", "section"],
)

answer_text = answer(openai, config["chat_deployment"], question, hits, cite=True)
print(answer_text)

Full-time employees accrue 15 days of PTO per calendar year. After five years of continuous service, this increases to 20 days per year [1].


## 3. Refusal + citation verification — same one-liners

Day 3's two-layer refusal and Day 6's LLM-as-judge verification are now single function calls.

In [6]:
# Day 3: refusal — try a question whose answer is NOT in the documents
result = answer_with_refusal(
    openai, config["chat_deployment"],
    question="What is the company's stock price?",
    hits=retrieve(search, openai,
                  "What is the company's stock price?",
                  config["embedding_deployment"], k=5,
                  select=["content", "source", "page", "section"]),
)
print(f"status: {result['status']}")
print(f"answer: {result['answer']}\n")

# Day 6: citation verification
verifications = verify_all(openai, config["chat_deployment"], answer_text, hits)
print(f"Citations parsed: {len(verifications)}")
for v in verifications:
    print(f"  [{v['cite']}] {v['verdict']:<11} — '{v['sentence'][:70]}'")

status: refused-low-confidence
answer: I don't know based on the documents.

Citations parsed: 1
  [1] SUPPORTED   — 'After five years of continuous service, this increases to 20 days per '


## Day 7 checklist

1. `common/` module created and importable from any day folder
2. This notebook ran a Day-6-class query end-to-end using only `common/` imports
3. Top-level files in place: `README.md`, `requirements.txt`, `.env.example`, `.gitignore`
4. `git status` shows `.env` is ignored
5. Repository pushed to GitHub
6. Per-day READMEs (start with this one, replicate for Days 2–6 over time)

### Learnings:
1. **Shared utilities make every future day cheaper.** Adding Day 8 (hybrid search) means writing ~20 lines, not 200.
2. **Refactoring is teaching.** Extracting the duplicated code into modules forced clear names: `condense_query`, `verify_all`, `build_filter`. Those names are now the vocabulary of the project.
3. **Secrets management is a design choice.** `.gitignore` is not optional. Treat the moment you touch a real API key as the moment your repo becomes security-sensitive.
